# Purpose

Prove the P2C commercial growth and experience activation is bounded, deterministic, and complete.

## Lemma: L74

In [ ]:
from pathlib import Path
import yaml

repo_root = Path.cwd()

agents = yaml.safe_load((repo_root / 'registry' / 'agents.yaml').read_text(encoding='utf-8'))
persona_reg = yaml.safe_load((repo_root / 'registry' / 'persona_registry_v2.yaml').read_text(encoding='utf-8'))
cap_map = yaml.safe_load((repo_root / 'registry' / 'persona_capability_map.yaml').read_text(encoding='utf-8'))

P2C = {
    'content-creator': (8, 'marketing'),
    'growth-hacker': (10, 'marketing'),
    'ui-designer': (7, 'design'),
    'brand-guardian': (8, 'design'),
    'feedback-synthesizer': (7, 'product'),
    'sprint-prioritizer': (8, 'product'),
    'ux-researcher': (7, 'design'),
    'studio-producer': (9, 'project-management'),
}

agents_by_name = agents['agents']
personas_by_id = persona_reg['personas']

for pid, (expected_actions, dept) in P2C.items():
    # Registry-backed
    p = personas_by_id[pid]
    assert p['coverage_status'] == 'registry-backed', f'{pid} not registry-backed'
    assert p['status'] == 'active', f'{pid} not active'
    # Action-surfaced with correct count
    a = agents_by_name[pid]
    assert len(a['allowed_actions']) == expected_actions, f'{pid} action count mismatch: {len(a["allowed_actions"])} != {expected_actions}'
    # Capability map parity
    c = cap_map['departments'][dept]['personas'][pid]
    assert c['coverage_status'] == 'registry-backed', f'{pid} cap map not registry-backed'
    assert c['current_action_count'] == expected_actions, f'{pid} cap map count mismatch'

# Total new actions = 64
total = sum(n for n, _ in P2C.values())
assert total == 64, f'Expected 64 total actions, got {total}'

# Department packs exist and are updated
for dept_file in ['marketing', 'design', 'product', 'project_management']:
    path = repo_root / 'registry' / f'department_pack_{dept_file}_v1.yaml'
    assert path.exists(), f'Missing department pack: {path}'

# Routing coverage: all actions route via prefix_routes
routing = yaml.safe_load((repo_root / 'registry' / 'executor_routing_v2.yaml').read_text(encoding='utf-8'))
prefix_routes = routing['prefix_routes']
for pid, (_, dept) in P2C.items():
    a = agents_by_name[pid]
    for action in a['allowed_actions']:
        prefix = action.split('.')[0] + '.'
        assert prefix in prefix_routes, f'{pid} action {action} has no routing prefix'

print('ALL L74 CHECKS PASSED')